# Composite Vertical Profiles for O3, T, and U

In this notebook, we:
1. **Read** the multi-year merged NetCDF file to obtain a **climatology** (annual mean cycle) for each variable.
2. **Extract** a single year's data from the merged file (the 'reference' year), then compute **(YearData - Climatology)**.
3. **Extract** each experiment's data (e.g., `0008Feb_small`), then also compute **(Experiment - Climatology)**.
4. **Plot** these anomalies in a composite figure, including all experiments plus the reference. The X-axis (months) is trimmed to start from the latest start month among all runs and ends in May.

**Key points:**
- O3 is averaged over longitude and lat 60°–90°N (cosine-weighted).
- T is averaged over longitude and then we take the minimum from lat 60°–90°N.
- U is averaged over longitude and then we pick the value at lat=60°N (nearest).
- Each subplot's data is already `Data - Climatology`, so the plotting function only needs to plot these anomalies directly.


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import math
import cftime

# ----------------------------------------------------
# Global paths and file references
# ----------------------------------------------------
MERGED_FILE = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'

# ----------------------------------------------------
# Data Processing Functions
# ----------------------------------------------------
def process_variable(ds, var):
    """
    对 O3、T、U 进行预处理：
    - O3：对经度求平均，再在 60–90°N 进行余弦加权平均；
    - T：对经度求平均，再在 60–90°N 选取并取最小值；
    - U：对经度求平均，然后选取 60°N 附近的值（nearest）。
    """
    if var == 'O3':
        da = ds['O3'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=slice(60, 90))
        weights = np.cos(np.deg2rad(da.lat))
        da = da.weighted(weights).mean(dim='lat')
    elif var == 'T':
        da = ds['T'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=slice(60, 90)).min(dim='lat')
    elif var == 'U':
        da = ds['U'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=60, method='nearest')
    else:
        raise ValueError("Unsupported variable: " + var)
    return da

def create_daily_data(da, start_month=1, end_month=5):
    """
    筛选 start_month 到 end_month 内的时间，返回每日数据。
    da = da.transpose('plev', 'time')会返回 (time_vals, plev_vals, data_array)，数据形状为 (plev, time)。
    """
    da = da.sel(time=da.time.dt.month.isin(range(start_month, end_month+1)))
    da = da.transpose('plev', 'time')
    time_vals = da['time'].values
    plev_vals = da['plev'].values
    data_array = da.values
    return time_vals, plev_vals, data_array

def read_climatology(var):
    """
    读取 MERGED_FILE 中所有年份的数据，计算 1-5 月每日平均值作为气候态。
    返回 dims=(plev, time) 的 DataArray。
    """
    ds = xr.open_dataset(MERGED_FILE)
    da = process_variable(ds, var)
    ds.close()
    # 筛选 1-5 月数据
    da = da.sel(time=da.time.dt.month.isin(range(1, 6)))
    calendar = ds.time.encoding.get('calendar', 'standard')
    # 计算所有年份的每日平均值
    da_clim = da.groupby('time.dayofyear').mean(dim='time')
    # 生成虚拟时间坐标（2000年 1-5 月）
    da_clim = da_clim.transpose('plev', 'dayofyear')
    #print(da.dims)  # Should be ('plev', 'time')
    #print(da.shape)  # Should be (23, <some_time_length>)
    days = da_clim['dayofyear'].values
    # 使用与实验数据相同的日历生成时间数组（参考年份设为2000）
    time_vals = [cftime.num2date(d - 1, units='days since 2000-01-01', calendar=calendar) for d in days]
    plev_vals = da_clim['plev'].values
    data_arr = da_clim.values
    clim_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return clim_da

def read_year_data(var, year):
    """
    读取 MERGED_FILE 中指定年份的 1-5 月数据。
    返回 dims=(plev, time) 的 DataArray。
    """
    ds = xr.open_dataset(MERGED_FILE)
    ds_year = ds.sel(time=ds.time.dt.year == int(year))
    da = process_variable(ds_year, var)
    ds.close()
    time_vals, plev_vals, data_arr = create_daily_data(da, 1, 5)
    year_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return year_da

def read_experiment_data(var, file_pattern, start_month=1):
    """
    打开实验数据文件，处理变量后返回从 start_month 到 5 月的每日数据。
    返回 dims=(plev, time) 的 DataArray。
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    da = process_variable(ds, var)
    ds.close()
    time_vals, plev_vals, data_arr = create_daily_data(da, start_month, 5)
    exp_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return exp_da


def get_anomaly(data_da, clim_da):
    # 将时间转换为 "%m-%d" 格式，忽略年份
    data_time_str = data_da.time.dt.strftime("%m-%d").values
    clim_time_str = clim_da.time.dt.strftime("%m-%d").values
    
    # 找到共同的月份和日子
    common_time_str = np.intersect1d(data_time_str, clim_time_str)
    
    # 选择 data_da 中对应共同月份和日子的数据
    data_da_common = data_da.sel(time=data_da.time.dt.strftime("%m-%d").isin(common_time_str))
    
    # 选择 clim_da 中对应共同月份和日子的数据
    clim_da_common = clim_da.sel(time=clim_da.time.dt.strftime("%m-%d").isin(common_time_str))
    
    # 计算异常值
    anom_vals = data_da_common.values - clim_da_common.values
    
    # 创建异常值的 DataArray
    anom_da = xr.DataArray(
        anom_vals,
        coords={'plev': data_da_common.plev, 'time': clim_da_common.time},
        dims=['plev', 'time']
    )
    return anom_da
# ----------------------------------------------------
# Plotting Functions
# ----------------------------------------------------
def auto_subplot_layout(n):
    """返回 (rows, cols) 用于 n 个子图，尽可能接近正方形排列。"""
    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))
    return rows, cols

def plot_composite(runs, ref_run, clim_da, var, composite_name, year, save_path):
    """
    绘制所有 run 和 REF 的异常场，叠加气候态等值线。
    所有变量使用手动指定的等值线，X 轴显示月份名称。
    """
    all_runs = runs + [ref_run]

    # 1) 确定共同时间
    common_time = None
    for run in all_runs:
        if common_time is None:
            common_time = run['time']
        else:
            common_time = np.intersect1d(common_time, run['time'])
    
    # 同步裁剪时间和数据
    for r in all_runs:
        mask = np.isin(r['time'], common_time)
        r['time'] = r['time'][mask]
        r['data'] = r['data'][:, mask]
    
    clim_mask = np.isin(clim_da.time.values, common_time)
    clim_run = {
        'time': clim_da.time.values[clim_mask].copy(),
        'plev': clim_da.plev.values.copy(),
        'data': clim_da.values[:, clim_mask].copy()
    }

    # 2) 统一异常值色阶
    all_anom = np.concatenate([r['data'].ravel() for r in all_runs])
    vmax = np.nanmax(np.abs(all_anom))
    vmin = -vmax

    # 3) 设置子图
    total_subplots = len(all_runs)
    rows, cols = auto_subplot_layout(total_subplots)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), squeeze=False)

    # 4) 定义等值线水平和标注格式
    if var == 'O3':
        clim_levels = [4.8e-6, 5.2e-6, 5.6e-6, 6.0e-6, 6.4e-6]
        fmt = '%.1e'  # 科学计数法，保留一位小数
    elif var == 'T':
        clim_levels = [204, 216, 228, 240, 252, 264]
        fmt = '%d'    # 整数
    elif var == 'U':
        clim_levels = [-8, 2, 12, 22, 32, 42]
        fmt = '%d'    # 整数
    else:
        raise ValueError("Unsupported variable: " + var)

    subplot_idx = 0
    cf = None
    for run in all_runs:
        r_idx = subplot_idx // cols
        c_idx = subplot_idx % cols
        ax = axes[r_idx][c_idx]
        if len(run['time']) > 0:
            time_mpl = mdates.date2num(run['time'])
            cf = ax.contourf(time_mpl, run['plev'], run['data'], levels=20,
                             cmap='RdBu_r', vmin=vmin, vmax=vmax)
            
            # 叠加气候态等值线（手动指定水平）
            clim_data = clim_run['data']
            CS = ax.contour(mdates.date2num(clim_run['time']), clim_run['plev'], clim_data,
                            levels=clim_levels, colors='k', linewidths=1.5)
            ax.clabel(CS, inline=True, fontsize=8, fmt=fmt)  # 使用指定的格式
            
            ax.set_yscale('log')
            ax.invert_yaxis()
            ax.set_xlim(time_mpl[0], time_mpl[-1])
            ax.xaxis.set_major_locator(mdates.MonthLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))  # 显示 "Feb", "Mar" 等
        ax.set_ylabel('Pressure (Pa)', fontsize=8)
        ax.set_title(f"{run['label']} ({run['year']})", fontsize=9)
        subplot_idx += 1

    # 隐藏多余子图
    for i in range(subplot_idx, rows*cols):
        r_idx = i // cols
        c_idx = i % cols
        axes[r_idx][c_idx].axis('off')
    
    if cf is not None:
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(cf, cax=cax, label=f'{var} Anomaly')
    
    fig.suptitle(f"{composite_name} {var} Anomaly, Year {year}", fontsize=12)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"[INFO] Saved composite figure: {save_path}")

# ----------------------------------------------------
# 定义 composite groups（20组）及辅助构造文件路径
# ----------------------------------------------------
def get_year_small_pert(year):
    base = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    return file_Feb, file_Mar

def get_year_nocouple(year):
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    return file_Feb, file_Mar

file_0008_Jan_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'
file_0008_Feb_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Feb_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Apr_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/BWCN.e122.f19_g16.002.0008-04.*.nc'
file_0008_Feb_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc'

composite_groups = [
    {'name': 'Composite 1', 'year': '0008', 'runs': [
        {'label': 'Jan small pertlim', 'file': file_0008_Jan_small, 'start_month': 1},
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small, 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small, 'start_month': 3}
    ]},
    {'name': 'Composite 2', 'year': '0008', 'runs': [
        {'label': 'Feb large pertlim', 'file': file_0008_Feb_large, 'start_month': 2},
        {'label': 'Mar large pertlim', 'file': file_0008_Mar_large, 'start_month': 3},
        {'label': 'Apr large pertlim', 'file': file_0008_Apr_large, 'start_month': 4}
    ]},
    {'name': 'Composite 3', 'year': '0008', 'runs': [
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small, 'start_month': 2},
        {'label': 'Feb large pertlim', 'file': file_0008_Feb_large, 'start_month': 2},
        {'label': 'Feb moist pertlim', 'file': file_0008_Feb_moist, 'start_month': 2}
    ]},
    {'name': 'Composite 4', 'year': '0008', 'runs': [
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small, 'start_month': 3},
        {'label': 'Mar large pertlim', 'file': file_0008_Mar_large, 'start_month': 3},
        {'label': 'Mar moist pertlim', 'file': file_0008_Mar_moist, 'start_month': 3}
    ]},
    {'name': 'Composite 5', 'year': '0003', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0003')[0], 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0003')[1], 'start_month': 3}
    ]},
    {'name': 'Composite 6', 'year': '0013', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0013')[0], 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0013')[1], 'start_month': 3}
    ]},
    {'name': 'Composite 7', 'year': '0014', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0014')[0], 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0014')[1], 'start_month': 3}
    ]},
    {'name': 'Composite 8', 'year': '0019', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0019')[0], 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0019')[1], 'start_month': 3}
    ]},
    {'name': 'Composite 9', 'year': '0008', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc', 'start_month': 2},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 10', 'year': '0013', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc', 'start_month': 2},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 11', 'year': '0014', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc', 'start_month': 2},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 12', 'year': '0019', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc', 'start_month': 2},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 13', 'year': '0008', 'runs': [
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small, 'start_month': 2},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc', 'start_month': 2}
    ]},
    {'name': 'Composite 14', 'year': '0008', 'runs': [
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small, 'start_month': 3},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 15', 'year': '0013', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0013')[0], 'start_month': 2},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc', 'start_month': 2}
    ]},
    {'name': 'Composite 16', 'year': '0013', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0013')[1], 'start_month': 3},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 17', 'year': '0014', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0014')[0], 'start_month': 2},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc', 'start_month': 2}
    ]},
    {'name': 'Composite 18', 'year': '0014', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0014')[1], 'start_month': 3},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 19', 'year': '0019', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0019')[0], 'start_month': 2},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc', 'start_month': 2}
    ]},
    {'name': 'Composite 20', 'year': '0019', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0019')[1], 'start_month': 3},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc', 'start_month': 3}
    ]}
]

def generate_all_composites():
    output_dir = '/home/weiji/restart_exam/plots/Vertical_Profiles_new/'
    variables = ['O3', 'T', 'U']
    for var in variables:
        print(f"\n[INFO] Generating composite plots for {var}...")
        clim_da = read_climatology(var)
        for group in composite_groups:
            year = group['year']
            composite_name = group['name']
            runs_info = group['runs']
            # 读取 REF 数据并计算异常值
            ref_da = read_year_data(var, year)
            ref_anom = get_anomaly(ref_da, clim_da)
            ref_run = {
                'label': 'REF',
                'year': year,
                'time': ref_anom.time.values,
                'plev': ref_anom.plev.values,
                'data': ref_anom.values
            }
            # 读取实验数据并计算异常值
            exp_runs = []
            for r in runs_info:
                exp_da = read_experiment_data(var, r['file'], r['start_month'])
                exp_anom = get_anomaly(exp_da, clim_da)
                exp_runs.append({
                    'label': r['label'],
                    'year': year,
                    'time': exp_anom.time.values,
                    'plev': exp_anom.plev.values,
                    'data': exp_anom.values
                })
            save_path = os.path.join(output_dir, f"{composite_name.replace(' ', '_')}_{var}.png")
            plot_composite(exp_runs, ref_run, clim_da, var, composite_name, year, save_path)
    print("\n[INFO] All composite plots have been generated.")

# 运行生成复合图的函数
generate_all_composites()

def plot_climatology(clim_da, var, save_path):
    """
    绘制气候态臭氧的垂直剖面填色图，与异常图设置一致。
    """
    fig, ax = plt.subplots(figsize=(6, 4))
    time_mpl = mdates.date2num(clim_da.time.values)  # 将时间转换为 matplotlib 可识别格式
    cf = ax.contourf(time_mpl, clim_da.plev, clim_da.values, levels=20, cmap='viridis')  # 绘制填色图
    ax.set_yscale('log')  # Y 轴为对数刻度
    ax.invert_yaxis()  # 反转 Y 轴（压力从大到小）
    ax.set_xlim(time_mpl[0], time_mpl[-1])  # 设置 X 轴范围
    ax.xaxis.set_major_locator(mdates.MonthLocator())  # X 轴显示每月
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))  # X 轴显示 "Jan", "Feb" 等
    ax.set_ylabel('Pressure (Pa)', fontsize=10)  # Y 轴标签
    ax.set_title(f'Climatology of {var}', fontsize=12)  # 标题
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # 添加色条位置
    fig.colorbar(cf, cax=cax, label=f'{var} Climatology')  # 添加色条
    plt.savefig(save_path, dpi=150, bbox_inches='tight')  # 保存图形
    plt.close(fig)  # 关闭图形，避免内存占用
    print(f"[INFO] Saved climatology figure: {save_path}")

# 调用函数（假设在主程序或 generate_all_composites 中）
clim_da = read_climatology('O3')  # 获取气候态臭氧数据
output_dir = '/home/weiji/restart_exam/plots/Vertical_Profiles_new/'
save_path_clim = os.path.join(output_dir, "Climatology_O3.png")  # 设置保存路径
plot_climatology(clim_da, 'O3', save_path_clim)  # 绘制并保存